# VF Logistics - Snowpark Python Risk Scoring & Workflow Orchestration

**Purpose**: Demonstrates Python/Snowpark usage for hackathon Judging Criteria (Section 9, item #2: Python/Java/Scala; item #4: Snowpark bonus).

This notebook:
1. Computes composite risk scores using Python/Snowpark DataFrame API
2. Triggers the full CLI-executable agentic workflow from Python
3. Proves the workflow is interface-agnostic (SQL CLI, CoCo CLI, or Python/Snowpark)

---

## Setup: Import Libraries & Get Session

In [ ]:
# Import Snowpark
from snowflake.snowpark import Session
from snowflake.snowpark.functions import col, when, lit
from snowflake.snowpark.context import get_active_session

# Get active session (Snowflake Notebook automatically provides this)
session = get_active_session()
print(f"✅ Connected to Snowflake account: {session.get_current_account()}")
print(f"   Database: {session.get_current_database()}")
print(f"   Schema: {session.get_current_schema()}")
print(f"   Warehouse: {session.get_current_warehouse()}")

## Step 1: Compute Composite Risk Score (Python Logic)

In [ ]:
def compute_composite_risk_score(session: Session):
    """
    Python-side reasoning layer: joins BILL_OF_LADING + FRAUD_ALERT and
    computes a composite risk score using weighted business rules that are
    easier to iterate on in Python than in pure SQL (e.g. for future ML
    model integration via Snowpark ML).
    """
    bl = session.table("MENDIX_APP.AGENTS.BILL_OF_LADING")
    alerts = session.table("MENDIX_APP.AGENTS.FRAUD_ALERT")

    joined = (
        bl.join(alerts, bl["BL_ID"] == alerts["BL_ID"], "inner")
        .select(
            bl["BL_ID"],
            bl["BL_NUMBER"],
            bl["SHIPPER_NAME"],
            bl["TOTAL_CHARGES"],
            bl["GROSS_WEIGHT_KGS"],
            alerts["ALERT_ID"],
            alerts["ALERT_TYPE"],
            alerts["SEVERITY"],
            alerts["STATUS"],
        )
    )

    # Composite risk score: weighted combination of signals (Python business logic)
    scored = joined.with_column(
        "PY_RISK_SCORE",
        (
            when(col("SEVERITY") == "HIGH", lit(50))
            .when(col("SEVERITY") == "MEDIUM", lit(25))
            .otherwise(lit(5))
        )
        + when(col("TOTAL_CHARGES") > 50000, lit(30)).otherwise(lit(0))
        + when(col("GROSS_WEIGHT_KGS") > 30000, lit(15)).otherwise(lit(0)),
    ).with_column(
        "PY_RISK_TIER",
        when(col("PY_RISK_SCORE") >= 80, lit("CRITICAL"))
        .when(col("PY_RISK_SCORE") >= 50, lit("HIGH"))
        .when(col("PY_RISK_SCORE") >= 25, lit("MEDIUM"))
        .otherwise(lit("LOW")),
    )

    return scored.sort(col("PY_RISK_SCORE").desc())


# Execute the risk scoring
print("Computing composite risk scores via Python/Snowpark...\n")
scored_df = compute_composite_risk_score(session)

# Show top 10 highest-risk shipments
scored_df.show(10)

## Step 2: Analyze Risk Distribution

In [ ]:
# Count shipments by risk tier
risk_distribution = scored_df.group_by("PY_RISK_TIER").count().sort("COUNT", ascending=False)

print("Risk Distribution (Python-computed tiers):\n")
risk_distribution.show()

# Highlight CRITICAL tier
critical_count = scored_df.filter(col("PY_RISK_TIER") == "CRITICAL").count()
print(f"\n🚨 CRITICAL-tier shipments identified by Python scoring layer: {critical_count}")

## Step 3: Trigger Full Agentic Workflow from Python

In [ ]:
def run_full_workflow_from_python(session: Session) -> str:
    """
    Invokes the same CLI-executable orchestrator (WORKFLOW_FULL_PIPELINE_V2)
    directly from Python via Snowpark -- proving the agentic workflow is
    callable from any interface: SQL CLI, CoCo CLI, or Python/Snowpark.
    """
    result = session.sql(
        "CALL MENDIX_APP.AGENTS.WORKFLOW_FULL_PIPELINE_V2('AUTO')"
    ).collect()
    return result[0][0]


print("🤖 Triggering full agentic workflow from Python (Snowpark)...\n")
import time
start_time = time.time()

result = run_full_workflow_from_python(session)

elapsed = time.time() - start_time
print(f"\n✅ Workflow completed in {elapsed:.2f} seconds")
print(f"\nResult: {result}")

## Step 4: Verify Audit Trail

In [ ]:
# Query the audit trail to prove multi-step orchestration
audit_trail = session.sql("""
    SELECT 
        AUDIT_ID,
        STEP_ORDER,
        STEP_NAME,
        STATUS,
        EXECUTED_AT
    FROM MENDIX_APP.AGENTS.WORKFLOW_AUDIT_LOG
    WHERE WORKFLOW_NAME = 'FULL_PIPELINE_V2'
    ORDER BY AUDIT_ID DESC
    LIMIT 5
""")

print("📋 Audit Trail (Last 5 steps):")
audit_trail.show()

## Summary

✅ **Demonstrated**:
1. Python/Snowpark DataFrame API for composite risk scoring
2. Business logic easier to express in Python than SQL
3. Full agentic workflow callable from Python (interface-agnostic architecture)
4. Complete audit trail proving multi-step orchestration

**Hackathon Compliance**:
- ✅ Section 9, Criterion #2: Python usage
- ✅ Section 9, Criterion #4: Snowpark bonus
- ✅ Workflow executed through multiple interfaces (SQL CLI, CoCo CLI, Python/Snowpark)